In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

In [ ]:
df = pd.read_csv("Loan_Default.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
print(df["Status"].value_counts(dropna=False))
print(df["Status"].value_counts(normalize=True, dropna=False).mul(100).round(2))

In [ ]:
df = df.drop(columns=["ID", "year"], errors="ignore")
invalid_filters = {
    "Security_Type": "Indriect",
    "open_credit": "opc",
    "construction_type": "mh",
    "Secured_by": "opc"
}
for col, bad_value in invalid_filters.items():
    if col in df.columns:
        df = df[df[col].astype(str).str.strip().str.lower() != str(bad_value).strip().lower()]
df = df.drop(
    columns=["open_credit", "construction_type", "Secured_by", "Security_Type"],
    errors="ignore"
)
df = df.dropna(subset=["Status"]).reset_index(drop=True)
print("Shape after cleaning:", df.shape)
print("\nMissing values in target:", df["Status"].isna().sum())


In [ ]:
x = df.drop(columns=["Status"])
y = df["Status"].astype(int)
xtr, xte, ytr, yte = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)
print("Training shape:", xtr.shape)
print("Testing shape :", xte.shape)


In [ ]:
outlier_cols = [
    "loan_amount",
    "rate_of_interest",
    "Interest_rate_spread",
    "Upfront_charges",
    "term",
    "property_value",
    "income",
    "LTV"
]
outlier_cols = [c for c in outlier_cols if c in xtr.columns]
def get_iqr_limits(train_df, columns):
    limits = {}
    for col in columns:
        series = pd.to_numeric(train_df[col], errors="coerce")
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        limits[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return limits
iqr_limits = get_iqr_limits(xtr, outlier_cols)
def cap_using_limits(data, limits):
    data = data.copy()
    for col, (low, high) in limits.items():
        data[col] = pd.to_numeric(data[col], errors="coerce").clip(low, high)
    return data
xtr = cap_using_limits(xtr, iqr_limits)
xte = cap_using_limits(xte, iqr_limits)
print("Outlier-capped numerical columns:", outlier_cols)


In [ ]:
num_cols = xtr.select_dtypes(include=np.number).columns.tolist()
cat_cols = xtr.select_dtypes(exclude=np.number).columns.tolist()
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
prep = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])
print("Numerical features:", len(num_cols))
print("Categorical features:", len(cat_cols))


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("prep", prep),
        ("model", RandomForestClassifier(
            n_estimators=250,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}
res = []
fitted_models = {}
for name, pipeline in models.items():
    print(f"Training {name}...")
    pipeline.fit(xtr, ytr)
    pred = pipeline.predict(xte)
    prob = pipeline.predict_proba(xte)[:, 1]
    res.append({
        "Model": name,
        "Accuracy": accuracy_score(yte, pred),
        "Precision": precision_score(yte, pred, zero_division=0),
        "Recall": recall_score(yte, pred, zero_division=0),
        "F1": f1_score(yte, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(yte, prob)
    })
    fitted_models[name] = pipeline
results_df = pd.DataFrame(res).sort_values("F1", ascending=False).reset_index(drop=True)
display(results_df.round(4))


In [ ]:
best_baseline_name = results_df.loc[0, "Model"]
best_baseline = fitted_models[best_baseline_name]
print("Best baseline model:", best_baseline_name)


In [ ]:
logistic_pipeline = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        random_state=RANDOM_STATE
    ))
])
param_distributions = {
    "model__C": np.logspace(-2, 2, 12),
    "model__solver": ["liblinear", "lbfgs"],
}
search = RandomizedSearchCV(
    estimator=logistic_pipeline,
    param_distributions=param_distributions,
    n_iter=8,
    scoring="f1",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
search.fit(xtr, ytr)
print("Best parameters:", search.best_params_)
print("Best cross-validation F1:", round(search.best_score_, 4))
tuned_model = search.best_estimator_


In [ ]:
final_pred = tuned_model.predict(xte)
final_prob = tuned_model.predict_proba(xte)[:, 1]
final_metrics = pd.DataFrame([{
    "Accuracy": accuracy_score(yte, final_pred),
    "Precision": precision_score(yte, final_pred, zero_division=0),
    "Recall": recall_score(yte, final_pred, zero_division=0),
    "F1": f1_score(yte, final_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(yte, final_prob)
}])
display(final_metrics.round(4))
print("\nClassification Report:\n")
print(classification_report(yte, final_pred, digits=4, zero_division=0))


In [ ]:
cm = confusion_matrix(yte, final_pred)
print("Confusion Matrix:")
print(cm)
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Default (0)", "Default (1)"]
).plot()
plt.title("Final Model - Confusion Matrix")
plt.show()


In [ ]:
RocCurveDisplay.from_predictions(yte, final_prob)
plt.title("Final Model - ROC Curve")
plt.show()


In [ ]:
tuned_row = {
    "Model": "Tuned Logistic Regression",
    "Accuracy": accuracy_score(yte, final_pred),
    "Precision": precision_score(yte, final_pred, zero_division=0),
    "Recall": recall_score(yte, final_pred, zero_division=0),
    "F1": f1_score(yte, final_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(yte, final_prob)
}
comparison = pd.concat(
    [results_df, pd.DataFrame([tuned_row])],
    ignore_index=True
).sort_values("F1", ascending=False).reset_index(drop=True)
display(comparison.round(4))


In [ ]:
print("Input columns:")
print(x.columns.tolist())


In [ ]:
sample = xte.iloc[[0]].copy()
prediction = tuned_model.predict(sample)[0]
probability = tuned_model.predict_proba(sample)[0, 1]
print("Predicted Status:", prediction)
print(f"Probability of default (Status=1): {probability:.2%}")
if prediction == 1:
    print("Result: Higher-risk / likely default")
else:
    print("Result: Lower-risk / likely non-default")


In [ ]:
MODEL_PATH = "loan_default_model.joblib"
joblib.dump(tuned_model, MODEL_PATH)
print(f"Model saved successfully: {MODEL_PATH}")


In [ ]:
loaded_model = joblib.load(MODEL_PATH)
loaded_pred = loaded_model.predict(sample)[0]
loaded_prob = loaded_model.predict_proba(sample)[0, 1]
print("Loaded model prediction:", loaded_pred)
print(f"Loaded model default probability: {loaded_prob:.2%}")
